# Day 1: Your First Graph

Welcome to Day 1. Today we're not going to touch an LLM or an API key at all. That's on purpose.

When people start with LangGraph they usually jump straight into agents, and then they get lost, because they're learning two things at once: how the LLM behaves *and* how the graph works. So we'll split it. For the first three days we learn the graph part with plain Python. Once that feels easy, adding an LLM on Day 4 is honestly just one extra line inside a function.

By the end of today you'll be able to:

- explain what State, Nodes and Edges are, in your own words
- build and run a small graph
- see what your graph looks like as a diagram
- watch it run step by step

Run each cell with `Shift + Enter`, top to bottom. And please, don't just run them. Change things, break things, see what happens. That's how this sticks.

## 1. So what is LangGraph?

Think about how a food delivery order works.

You place an order, the restaurant cooks it, a rider picks it up, and it gets delivered. There's a slip of paper (or a screen) that travels with the order. Each person reads it, does their job, and writes something on it: "cooked at 7:10", "picked up by Ravi", "delivered".

LangGraph works exactly like that:

- The **state** is that order slip. It's the shared data everyone can read and write.
- A **node** is one person doing one job. In code, it's just a Python function.
- An **edge** is the arrow that says who goes next.
- **START** and **END** mark where the flow begins and where it stops.

That's really all a graph is. A few functions, connected by arrows, passing one shared dictionary along.

**Why should you care?** If your program is just "do A, then B, then C", you don't need LangGraph, plain Python is fine. But the moment you need "do A, then *decide* between B and C, maybe loop back to A, wait for a human to approve, remember what happened yesterday"... plain functions get messy fast. LangGraph gives you a clean structure for exactly that. And that's what AI agents are: programs that decide, loop and remember.

## 2. Setup

Run this once. If you already did `pip install -r requirements.txt` from the main folder, you can skip it.

In [ ]:
%pip install -q -U langgraph

In [ ]:
from importlib.metadata import version

In [ ]:
print("LangGraph version:", version("langgraph"))

## 3. Hello World

Every graph you'll ever build in this course follows the same five steps. I want you to remember these, because we'll repeat them every single day:

1. Describe the **state** (what data travels through the graph)
2. Write the **nodes** (plain functions)
3. Create a **builder** and add the nodes to it
4. Connect the nodes with **edges**
5. **Compile** it and run it

Let's build the smallest graph possible. One node that says hello.

```
START  -->  greet  -->  END
```

First, the state. We use a `TypedDict`, which is just a normal dictionary where we write down which keys exist and what type they hold. Think of it as designing the order slip before anyone writes on it.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
class State(TypedDict):
    name: str
    greeting: str

Now the node. Look how ordinary this is. It's a function that takes the state, and returns a small dictionary with whatever it wants to change.

In [ ]:
def greet(state: State) -> dict:
    # Read the name from the slip, write a greeting back on it
    return {"greeting": f"Hello, {state['name']}! Welcome to LangGraph."}

Steps 3 and 4: we make a builder, register our node under a name, and draw the arrows.

In [ ]:
builder = StateGraph(State)

In [ ]:
builder.add_node("greet", greet)     # "greet" is the name we'll use in edges

In [ ]:
builder.add_edge(START, "greet")     # when the graph starts, go to greet
builder.add_edge("greet", END)       # after greet, we're done

### Predict before you run

We're going to call `app.invoke({"name": "Asha"})`. What do you think comes back? Just the greeting string? Or something else?

Make a guess, then run the next cell.

<details>
<summary>Click to see the answer</summary>

You get back the **whole state** as a dictionary, not just the greeting:

`{'name': 'Asha', 'greeting': 'Hello, Asha! Welcome to LangGraph.'}`

`invoke` always returns the final state. The `name` we passed in is still there, and the `greeting` the node wrote has been added to it.
</details>

In [ ]:
app = builder.compile()              # step 5: turn the drawing into something runnable

In [ ]:
result = app.invoke({"name": "Asha"})
print(result)

Let's slow down and walk through what just happened, because this is the core of everything:

1. `invoke({"name": "Asha"})` created the starting state: `{"name": "Asha"}`.
2. LangGraph followed the edge from START to `greet`, and called `greet(state)`.
3. `greet` returned `{"greeting": "..."}`. LangGraph took that and **merged** it into the state.
4. It followed the edge from `greet` to END, and handed you the final state.

Notice that the node never touched the state directly. It just returned what it wanted changed, and LangGraph did the updating. Hold on to that idea, it matters a lot on Day 2.

In [ ]:
# A quick self-check. If this cell runs without an error, your graph is correct.
assert result["greeting"] == "Hello, Asha! Welcome to LangGraph."

### Your turn

Change the name to your own, and change the greeting to something in your language. Then run it.

In [ ]:
def greet(state: State) -> dict:
    return {"greeting": f"Namaste, {state['name']}!"}    # try changing this message

In [ ]:
builder = StateGraph(State)

In [ ]:
builder.add_node("greet", greet)

In [ ]:
builder.add_edge(START, "greet")
builder.add_edge("greet", END)

In [ ]:
app = builder.compile()

In [ ]:
print(app.invoke({"name": "Asha"}))    # and put your name here

## 4. Seeing your graph

When graphs get bigger, it really helps to see them as a picture. A compiled graph can draw itself.

I'm going to write a small helper called `show_graph` that we'll reuse every day. It tries to draw a picture (that needs internet). If it can't, it prints the diagram as text, which you can paste into https://mermaid.live to see it.

In [ ]:
from IPython.display import Image, display

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        # No internet? No problem, print the text version instead
        print(app.get_graph().draw_mermaid())

In [ ]:
show_graph(app)

## 5. Two nodes in a row

One node is a bit boring. In real work you have steps, and each step builds on the previous one.

Here's a small text pipeline. The first node cleans up messy text, the second counts the words. The important bit: the second node uses what the first one wrote.

```
START  -->  clean  -->  count_words  -->  END
```

In [ ]:
class TextState(TypedDict):
    text: str
    clean_text: str
    word_count: int

In [ ]:
def clean(state: TextState) -> dict:
    # split() with no arguments throws away all the extra spaces,
    # then we join the words back with single spaces
    tidy = " ".join(state["text"].split())
    return {"clean_text": tidy.lower()}

In [ ]:
def count_words(state: TextState) -> dict:
    # clean_text exists because the clean node already ran
    return {"word_count": len(state["clean_text"].split())}

In [ ]:
builder = StateGraph(TextState)

In [ ]:
builder.add_node("clean", clean)
builder.add_node("count_words", count_words)

In [ ]:
builder.add_edge(START, "clean")
builder.add_edge("clean", "count_words")
builder.add_edge("count_words", END)

In [ ]:
text_app = builder.compile()

show_graph(text_app)

### Predict before you run

The input is `"   Hello     LangGraph   WORLD  "` (lots of extra spaces, mixed case). What will `clean_text` and `word_count` be?

<details>
<summary>Click to see the answer</summary>

`clean_text` will be `'hello langgraph world'` and `word_count` will be `3`.

The spaces don't count as words because `clean` already removed them before `count_words` ran.
</details>

In [ ]:
out = text_app.invoke({"text": "   Hello     LangGraph   WORLD  "})
print(out)

A small shortcut. When your nodes are just a straight line, you can use `add_sequence` instead of writing every edge yourself. The node names become the function names.

In [ ]:
builder = StateGraph(TextState)

In [ ]:
builder.add_sequence([clean, count_words])     # same as adding both nodes + the edge between them

In [ ]:
builder.add_edge(START, "clean")
builder.add_edge("count_words", END)

In [ ]:
app = builder.compile()

In [ ]:
print(app.invoke({"text": "one two   three four"}))

## 6. A node only returns what it changed

This is something students get wrong a lot in the first week, so let's be very clear about it.

A node does **not** return the whole state. It returns a small dictionary with only the keys it wants to change. Everything else is left exactly as it was.

It's like the restaurant writing "cooked at 7:10" on the slip. They don't rewrite the customer's address. They just add their part.

In [ ]:
class Profile(TypedDict):
    name: str
    age: int
    city: str

In [ ]:
def add_age(state: Profile) -> dict:
    return {"age": 30}          # only touches age

In [ ]:
def add_city(state: Profile) -> dict:
    return {"city": "Pune"}     # only touches city

In [ ]:
builder = StateGraph(Profile)

In [ ]:
builder.add_sequence([add_age, add_city])

In [ ]:
builder.add_edge(START, "add_age")
builder.add_edge("add_city", END)

In [ ]:
app = builder.compile()

In [ ]:
profile = app.invoke({"name": "Ravi"})
print(profile)

`name` was never returned by any node, and it's still there. Good.

## 7. Common mistakes

These are the ones I see in almost every batch.

**Mistake 1: Returning a key that isn't in the state.**
LangGraph doesn't complain. It just quietly ignores it, and then you spend twenty minutes wondering where your value went. Watch:

In [ ]:
def typo_node(state: Profile) -> dict:
    return {"citty": "Mumbai"}       # oops, "citty" is not a key in Profile

In [ ]:
builder = StateGraph(Profile)

In [ ]:
builder.add_node("typo_node", typo_node)

In [ ]:
builder.add_edge(START, "typo_node")
builder.add_edge("typo_node", END)

In [ ]:
app = builder.compile()

In [ ]:
print(app.invoke({"name": "Ravi", "city": "Pune"}))
# city is still Pune, and "citty" has simply disappeared. No error at all.

**Fix:** always add every key to your `TypedDict` first, and double-check spelling when a value "goes missing".

**Mistake 2: Forgetting the edge from START.**
If no edge leaves START, LangGraph doesn't know where to begin, so `compile()` fails.

In [ ]:
builder = StateGraph(Profile)

In [ ]:
builder.add_node("add_age", add_age)

In [ ]:
builder.add_edge("add_age", END)          # we forgot builder.add_edge(START, "add_age")

In [ ]:
try:
    builder.compile()
except ValueError as error:
    print("Error:", error)

**Fix:** every graph needs at least one edge out of START. If you see "entrypoint" in an error, this is almost always the reason.

**Mistake 3: Returning the whole state "just to be safe".**
It works for now, but on Day 2 you'll meet reducers, and then returning everything can duplicate your data. Get into the habit now: return only what changed.

## 8. Watching the graph run, step by step

`invoke()` only shows you the end result. When something goes wrong, you want to see what happened *in between*. That's what `stream()` is for.

There are two modes you'll use a lot:

- `stream_mode="updates"` shows what each node returned, one node at a time
- `stream_mode="values"` shows the full state after each step

In [ ]:
print("--- updates: what did each node change? ---")
for step in text_app.stream({"text": " LangGraph  is  FUN "}, stream_mode="updates"):
    print(step)

print()
print("--- values: what does the whole state look like after each step? ---")
for state in text_app.stream({"text": " LangGraph  is  FUN "}, stream_mode="values"):
    print(state)

### Predict before you run

With `stream_mode="values"`, how many lines got printed above, and why not just two (one per node)?

<details>
<summary>Click to see the answer</summary>

Three. The first one is the **starting** state, before any node ran. Then one after `clean` and one after `count_words`.

`"updates"` only printed two, because it only reports node outputs, and the starting input isn't a node.
</details>

## 9. Practice

Try these on your own first. The solutions are in the cells right after, but you'll learn much more if you struggle with it for five minutes before peeking.

**Exercise 1.** Build `START -> shout -> END`. The state has `text` and `loud`. The `shout` node should put the upper-case text followed by `"!!!"` into `loud`.

**Exercise 2.** Build a small billing pipeline with three nodes in a row:
- `add_tax` adds 18% GST to `price` and stores it in `total`
- `add_delivery` adds 40 rupees to `total`
- `round_total` rounds `total` to 2 decimals

For a price of 499, the total should come out to 628.82.

In [ ]:
# Solution 1
class ShoutState(TypedDict):
    text: str
    loud: str

In [ ]:
def shout(state: ShoutState) -> dict:
    return {"loud": state["text"].upper() + "!!!"}

In [ ]:
builder = StateGraph(ShoutState)

In [ ]:
builder.add_node("shout", shout)

In [ ]:
builder.add_edge(START, "shout")
builder.add_edge("shout", END)

In [ ]:
app = builder.compile()

In [ ]:
result = app.invoke({"text": "langgraph rocks"})
print(result)
assert result["loud"] == "LANGGRAPH ROCKS!!!"

In [ ]:
# Solution 2
class BillState(TypedDict):
    price: float
    total: float

In [ ]:
def add_tax(state: BillState) -> dict:
    return {"total": state["price"] * 1.18}

In [ ]:
def add_delivery(state: BillState) -> dict:
    return {"total": state["total"] + 40}

In [ ]:
def round_total(state: BillState) -> dict:
    return {"total": round(state["total"], 2)}

In [ ]:
builder = StateGraph(BillState)

In [ ]:
builder.add_sequence([add_tax, add_delivery, round_total])

In [ ]:
builder.add_edge(START, "add_tax")
builder.add_edge("round_total", END)

In [ ]:
app = builder.compile()

In [ ]:
bill = app.invoke({"price": 499})
print(bill)
assert bill["total"] == 628.82

## Wrapping up

Today you learned the three building blocks. The **state** is the shared slip of paper. **Nodes** are plain functions that read it and return only what they change. **Edges** decide the order. You compile the graph, run it with `invoke`, and look inside with `stream`.

If you can explain those three words to a friend without looking at this notebook, you're ready for Day 2.

Tomorrow we deal with an interesting problem: what happens when two nodes write to the *same* key? Does the second one wipe out the first? That's where **reducers** come in.